In [ ]:
import sys, os, shutil, json, math

import numpy as np
import papermill as pm
import pickle as pkl

from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.optimize import curve_fit

---
---
---

In [ ]:
def make_folder(name_folder):
    
    if not os.path.exists(name_folder): os.makedirs(name_folder)

---

In [ ]:
def execute_notebook_ALL(notebook_directory, notebook_filename, iteration, parameters, output_directory):
    
    '''
    Define a function to execute a notebook.
    
    Usage: This way, we can implement a multiprocessing pipeline which, when using jupyter notebooks, is much 
        faster than using built functions that paralellize the code. We thus write the code that we wish to run
        multiple parallel instances of into a separate notebook and run it from inside another notebook where it 
        needs to be executed.
    '''

    print(parameters)
    
    # Define the input and output paths for the notebook
    input_notebook_path  = os.path.join(notebook_directory, notebook_filename)
    output_notebook_path = os.path.join(output_directory, f"{notebook_filename}_output_{iteration}.ipynb")
    
    # Execute the notebook with papermill
    pm.execute_notebook(input_notebook_path, output_notebook_path, parameters=parameters)

In [ ]:
def Run_the_notebooks(notebook_filenames, notebook_parameters, notebook_directory, output_directory, max_parallel=3):
    
    '''
    Run notebooks in parallel, starting a new one whenever another finishes.
    '''
    
    with ThreadPoolExecutor(max_workers=max_parallel) as executor:
        # Submit each notebook execution as a separate future
        futures = [executor.submit(execute_notebook_ALL, notebook_directory, notebook_filename, i, notebook_parameters[i], output_directory)
                   for i, notebook_filename in enumerate(notebook_filenames)]
        
        # As each notebook completes, start a new one if available
        for future in as_completed(futures):
            future.result()  # This will raise any exceptions from the notebook execution

In [ ]:
def run_the_notebooks(running, notebook_filenames, notebook_parameters, notebook_directory, output_directory, cores_no):
    
    if running and (len(notebook_filenames) != 0): Run_the_notebooks(notebook_filenames, notebook_parameters, notebook_directory, output_directory, cores_no)
    else:
        for nbp in notebook_parameters: print(nbp)

---
---
---

In [ ]:
def deleter_of_DTFE_files(file_path_DTFE_Z, grid_chunks):
    
    for         ix in range(grid_chunks):
        for     iy in range(grid_chunks):
            for iz in range(grid_chunks):
                file_path_DTFE_Z_chunk = file_path_DTFE_Z+"chunk___"+str(ix)+"_"+str(iy)+"_"+str(iz)+"/"
    
                with open(file_path_DTFE_Z_chunk+"buffer.pk",       'wb') as f: pkl.dump(0, f)
                with open(file_path_DTFE_Z_chunk+"tree.pk",         'wb') as f: pkl.dump(0, f)
                with open(file_path_DTFE_Z_chunk+"coords.pk",       'wb') as f: pkl.dump(0, f)
                with open(file_path_DTFE_Z_chunk+"tetra_points.pk", 'wb') as f: pkl.dump(0, f)
                with open(file_path_DTFE_Z_chunk+"connections.pk",  'wb') as f: pkl.dump(0, f)
                with open(file_path_DTFE_Z_chunk+"rho.pk",          'wb') as f: pkl.dump(0, f)
                with open(file_path_DTFE_Z_chunk+"D_rho.pk",        'wb') as f: pkl.dump(0, f)
                np.save(  file_path_DTFE_Z_chunk+"Ainv_all.npy",                         0)
                np.save(  file_path_DTFE_Z_chunk+"origins.npy",                          0)

---
---
---

Functions to check the size of variables inside notebooks to better manage RAM.

Full credit to:
dinatrina. “Answer to ‘python- how to display size of all variables’” Stack Overflow, 19 Feb 2025,https://stackoverflow.com/questions/24455615/python-how-to-display-size-of-all-variables.

In [ ]:
def get_real_size(obj):
    """Recursively calculate the real memory size of an object."""
    size = sys.getsizeof(obj)
    if isinstance(obj, (list, tuple, set, frozenset)):
        size += sum(get_real_size(item) for item in obj)
    elif isinstance(obj, dict):
        size += sum(get_real_size(key) + get_real_size(value) for key, value in obj.items())
    return size

def get_memory_usage():
    # Separate user-defined and system variables
    user_vars = {k: v for k, v in globals().items() if not k.startswith('_') and not callable(v)}
    system_vars = {k: v for k, v in globals().items() if k.startswith('_') and not callable(v)}

    # Calculate memory usage using custom get_real_size function
    memory_usage = {k: get_real_size(v) for k, v in user_vars.items()}
    system_memory = sum(get_real_size(v) for v in system_vars.values())

    # Convert bytes to human-readable format
    def sizeof_fmt(num, suffix='B'):
        for unit in ['', 'K', 'M', 'G', 'T', 'P', 'E', 'Z']:
            if abs(num) < 1024.0:
                return f"{num:3.1f}{unit}{suffix}"
            num /= 1024.0
        return f"{num:.1f}Yi{suffix}"

    # Format memory usage
    formatted_usage = {k: sizeof_fmt(v) for k, v in memory_usage.items()}
    formatted_usage['_system_vars'] = sizeof_fmt(system_memory)

    # Sort by size in descending order and extract values
    sorted_sizes = sorted(formatted_usage.items(), key=lambda x: get_real_size(globals().get(x[0], 0)), reverse=True)

    # Print the sorted list
    for var, size in sorted_sizes:
        print("{:>30}: {:>8}".format(var, size))

    # Print totals
    print("\nTotal Memory Usage:")
    print(f"User Variables: {sizeof_fmt(sum(memory_usage.values()))}")
    print(f"System Variables: {sizeof_fmt(system_memory)}")
    print(f"Combined Total:     {sizeof_fmt(system_memory+sum(memory_usage.values()))}")

---
---
---